In [2]:
!export LANGSMITH_ENDPOINT="https://api.smith.langchain.com"
!export LANGSMITH_API_KEY="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"

In [9]:
from langsmith import Client
from collections import defaultdict
import json
from tqdm import tqdm

client = Client()

PROJECT_NAME = "YunjueAgent"   # 改成你的 project 名
OUTPUT_FILE = "project_traces.json"


def build_tree(runs):
    by_id = {}
    children_map = defaultdict(list)

    for r in tqdm(runs):
        rid = str(r.id)
        pid = str(r.parent_run_id) if r.parent_run_id else None

        by_id[rid] = {
            "id": rid,
            "trace_id": str(r.trace_id),
            "parent_run_id": pid,
            "name": r.name,
            "run_type": r.run_type,
            "inputs": r.inputs,
            "outputs": r.outputs,
            "error": r.error,
            "start_time": str(r.start_time),
            "end_time": str(r.end_time),
            "children": []
        }

        children_map[pid].append(rid)

    def attach(run_id):
        node = by_id[run_id]
        node["children"] = [attach(cid) for cid in children_map.get(run_id, [])]
        return node

    # 找 root（没有 parent 的）
    roots = children_map[None]
    return [attach(rid) for rid in roots]


print("Downloading runs...")

all_runs = list(
    client.list_runs(
        project_name=PROJECT_NAME,
        select=[
            "id",
            "trace_id",
            "parent_run_id",
            "name",
            "run_type",
            "inputs",
            "outputs",
            "error",
            "start_time",
            "end_time",
        ],
    )
)

print(f"Total runs downloaded: {len(all_runs)}")

print("Building trace tree...")

tree = build_tree(all_runs)

with open(OUTPUT_FILE, "w") as f:
    json.dump(tree, f, indent=2)

print(f"Saved to {OUTPUT_FILE}")

Total runs downloaded: 3697
Building trace tree...


100%|██████████| 3697/3697 [00:00<00:00, 150453.52it/s]


Saved to project_traces.json


In [2]:
import json
# load json file
with open("project_traces.json", "r") as f:
    data = json.load(f)

# print(data)

def is_target_task(name):
    last_part = name.split("-")[-1]
    # last_part can be transformed to int
    try:
        last_part = int(last_part)
        return last_part < 20
    except:
        return False

# only safe the trace with name "YunjueAgent-task-i" (i < 20)
data = [d for d in data if d["name"].startswith("YunjueAgent-task-") and is_target_task(d["name"])]

# save the data to a new json file
with open("project_traces_task_filter.json", "w") as f:
    json.dump(data, f, indent=2)

